<a href="https://colab.research.google.com/github/Shemuel-bot/AI-ball-notebooks/blob/main/ai_ball_core_variant_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Ball — Ball Trap Brain

A toy "ball trap" symbolic model: input characters are tokenized against an alphabet, turned into a ball trajectory through a ring of deflectors, and the deflectors the ball falls through ("holes") produce the output characters.

This notebook contains:
1. The tokenizer and the `Ball_Trap_Brain` model itself
2. Fitness evaluation utilities
3. A genetic algorithm (`main`) that evolves a single brain
4. A `Council_Brain` that combines several independently-trained brains by majority vote -- trained via **continuous relaxation** (`train_continuous`) instead of the genetic algorithm, voting with `predict_continuous` directly on each member's trained continuous parameters
5. The `Ball_Trap_Brain.train_continuous` continuous relaxation trainer itself (with optional curriculum learning), plus `predict_continuous` / `evaluate_continuous` / `commit_continuous` for using the trained continuous parameters directly
6. An optional, guarded cell that runs the full council training end to end


## Imports & tokenizer

In [ ]:
import random
import math
import copy

def TokenizeInput(input, alphabet):
    input_str = str(input)
    matches = []

    for order, item in enumerate(alphabet):
        item_str = str(item)
        if not item_str:
            continue

        # Loop to find ALL occurrences of the item instead of just the first
        start_pos = 0
        while True:
            index = input_str.find(item_str, start_pos)
            if index == -1:
                break

            matches.append((index, -len(item_str), order, item))

            # Move start position forward by 1 to continue searching the rest of the string
            # (Adding 1 instead of len(item_str) allows it to safely catch overlapping patterns)
            start_pos = index + 1

    # Sorts by starting index, then by longest string, then by original order
    matches.sort(key=lambda match: (match[0], match[1], match[2]))

    non_overlapping_items = []
    last_end_index = 0

    for match in matches:
        start_index = match[0]
        length = -match[1]  # The length was stored as a negative number
        item = match[3]

        # Only add the token if it doesn't overlap with the previously added token
        if start_index >= last_end_index:
            non_overlapping_items.append(item)
            last_end_index = start_index + length
    #print(non_overlapping_items)
    return non_overlapping_items

## `Ball_Trap_Brain`

Includes the original genetic-algorithm-oriented methods (`clone`, `Mutate`, `shift_alphabet`, ...) plus a gradient-based continuous relaxation trainer, split across a few methods:
- `train_continuous(...)`: trains the continuous parameters with gradient descent (see its docstring for details on the relaxation scheme). Temperature cools on a fixed linear schedule from `temp_start` to `temp_end` over `epochs` -- pass the same value for both to train at a fixed, non-annealed temperature instead. Returns `(loss_history, accuracy_history)`.
- `predict_continuous(input_str, ...)`: answer a question using the **continuous** brain directly, without discretizing anything -- no dependency on `deflectors_list` etc at all.
- `evaluate_continuous(test_cases, ...)`: like `evaluate_performance`, but scores `predict_continuous` instead of `compute_input`.
- `commit_continuous()`: discretize already-trained continuous parameters into the discrete brain without training any further.

**What gets trained vs. what doesn't:** every mutable field `Mutate` touches on the discrete brain -- `deflectors_list`, `alphabet_directions`, `alphabet_start_positions` -- plus `ball_energy` (the ball's starting energy, now a free real-valued scalar). `holes_multiplier`, `alphabet`, and `output` are **not** trained: `holes_multiplier` only controls discrete hole-assignment bookkeeping (`set_holes`) and never appears in the differentiable simulation, so there's no gradient path to it at all; `alphabet`/`output` are the literal vocabularies (which characters exist), not values assigned to them -- everything about *how* each character is used is already covered by the parameters above, and changing the vocabularies themselves would be a structural change, not a value to relax and optimize.

**Curriculum learning (`curriculum=True`):** instead of sampling every batch from the full `test_cases`, training starts on a small "active" subset (`curriculum_start_size` examples). Every `curriculum_check_every` epochs, the *real* exact-match accuracy on the active subset is measured with `evaluate_continuous` (not the cheap `token_acc` proxy); once it's >= `curriculum_threshold`, `curriculum_growth` more examples are folded in and training continues on the larger set, until the active subset covers everything. This keeps the periodic real-accuracy check cheap (it only ever runs against the small, growing active pool, not the full dataset) while still using an honest signal to decide when to grow.

**Two pollution bugs, now fixed, that affected discrete *and* continuous output alike:**
1. *Leading-position leakage.* The loss only scored a right-aligned tail of each trajectory; leading positions outside that window had no training signal at all and could leak arbitrary symbols into the real output. Fixed by folding "should this position emit at all" into the *same* per-position loss as a genuine "emit nothing" class -- `never_fired = 1 - out_dist.sum()` already forms a valid (K+1)-way distribution with the K real symbols, so every position (aligned or not) now has a well-defined target under one unified objective.
2. *Repeat/self-loop cycling.* Once a trajectory hit a hole, the "first hole reached" objective stopped caring what happened for the rest of its `cap` steps -- so a hole deflector could cycle with another hole (or itself) and just re-emit symbols for the whole `cap` budget (e.g. `"3333333333"`). Fixed with `repeat_penalty_weight` (default 0.3), which penalizes the expected number of *additional* hole-hits after the first one per trajectory.

**`commit` flag:** `train_continuous(..., commit=True)` (the default) discretizes the trained parameters back into this instance's own `deflectors_list`, `alphabet_directions` and `alphabet_start_positions`. Pass `commit=False` to leave the discrete brain completely untouched and only keep the trained continuous version -- this is what `train_council` (below) uses for every council member.

**How `token_acc` is scored (printed during training):** within each epoch's batch, for every trajectory position (aligned *or* blank), it checks whether the predicted class -- top symbol if emit-probability >= 0.5, else "blank" -- matches the target class, and reports the fraction that do. It's a cheap, free-to-compute *character-position* metric, not a *question* metric. For real, whole-question exact-match accuracy, use `evaluate_performance` (discrete) or `evaluate_continuous` (continuous).

In [ ]:
class Ball_Trap_Brain:
    def __init__(self, deflectors, bounce, alphabet, output, holes_multiplier=1):
        self.output = output
        self.holes_multiplier = holes_multiplier
        self.ball_energy = bounce*2
        self.alphabet = alphabet
        self.alphabet_directions = self.choose_alphabet_direction(len(alphabet))
        self.alphabet_start_positions = self.start_positions(len(alphabet), deflectors)
        self.deflectors_list = [[random.randint(0, deflectors-1), bounce] for i in range(deflectors)]
        self.set_holes(output, deflectors, self.holes_multiplier)

    def set_holes (self, output, deflectors, holes_multiplier):
        for i in range(len(output)):
          for j in range(holes_multiplier):
            self.deflectors_list[random.randint(0, deflectors-1)][1] = output[i]

    def choose_alphabet_direction(self, alphabet):
        directions = [random.choice([1,-1]) for _ in range(alphabet)]
        return directions

    def start_positions(self, alphabet, deflectors):
        return [random.randint(0, deflectors-1) for i in range(alphabet)]


    def compute_input(self, input):
        directions = []
        positions = []
        travel = []
        input_arr = TokenizeInput(input, self.alphabet)

        for i in input_arr:
            if i in self.alphabet:
                index = self.alphabet.index(i)
                directions.append(self.alphabet_directions[index])
                positions.append(self.alphabet_start_positions[index])

        for i in range(len(directions)):
            travel.append(self.travel(copy.copy(directions), positions[i], i))


        return self.ricochet(travel, 10, self.ball_energy)

    def ricochet(self, travel, cap, ball_energy):
        output = []
        dl = self.deflectors_list
        n = len(dl)

        for t in travel:
            # normalize index using modulo (handles large or negative values)
            new_idx = t % n
            cap_idx = 0
            be = ball_energy

            while be > 0 and cap_idx < cap:
                dest, val = dl[new_idx]
                distance = abs(dest - new_idx)

                if not isinstance(val, int):
                    output.append(val)
                    be += 1
                    be -= distance
                    new_idx = dest
                    cap_idx += 1
                    continue

                be += val
                be -= distance
                new_idx = dest
                cap_idx += 1

        return output

    def compute_deflectors(self, input):
        """
        Processes input exactly like compute_input but returns an array
        of all deflectors hit during the ball's travel.
        """
        directions = []
        positions = []
        travel = []
        input_arr = TokenizeInput(input, self.alphabet)

        for i in input_arr:
            if i in self.alphabet:
                index = self.alphabet.index(i)
                directions.append(self.alphabet_directions[index])
                positions.append(self.alphabet_start_positions[index])

        for i in range(len(directions)):
            travel.append(self.travel(copy.copy(directions), positions[i], i))

        # Passes the standard cap of 10 and the current ball_energy
        return self.ricochet_deflectors(travel, 10, self.ball_energy)

    def ricochet_deflectors(self, travel, cap, ball_energy):
        """
        Modified ricochet logic that records and returns the deflectors
        hit instead of the output values.
        """
        deflectors_hit = []
        dl = self.deflectors_list
        n = len(dl)

        for t in travel:
            # normalize index using modulo (handles large or negative values)
            new_idx = t % n
            cap_idx = 0
            be = ball_energy

            while be > 0 and cap_idx < cap:
                dest, val = dl[new_idx]

                # Append the deflector's details to our tracking array
                deflectors_hit.append(new_idx)

                distance = abs(dest - new_idx)

                if not isinstance(val, int):
                    be += 1
                    be -= distance
                    new_idx = dest
                    cap_idx += 1
                    continue

                be += val
                be -= distance
                new_idx = dest
                cap_idx += 1

        return deflectors_hit

    def travel(self, directions, position, index):
        directions.pop(index)
        position += sum(directions)
        return position

    def display_deflectors(self):
        for i, deflect in enumerate(self.deflectors_list):
            print(f"Deflector {i}: {deflect}")

    def clone(self):
      # Create a lightweight dummy instance
      new_brain = Ball_Trap_Brain(1, self.ball_energy // 2, self.alphabet, self.output, self.holes_multiplier)

      # Fast shallow copy of the 10,000 elements
      new_brain.deflectors_list = [[d, v] for d, v in self.deflectors_list]
      new_brain.alphabet_directions = self.alphabet_directions[:]
      new_brain.alphabet_start_positions = self.alphabet_start_positions[:]

      return new_brain

    def shift_alphabet(self):
        random.shuffle(self.alphabet)
        for i in range(len(self.deflectors_list)):
          if self.deflectors_list[i][1] in self.output:
            idx = self.output.index(self.deflectors_list[i][1])
            self.deflectors_list[i][1] = self.alphabet[idx]

    def Mutate(self, individual_probability=0.1, probability=0.05):
        if random.random() < individual_probability:
            for i in range(len(self.deflectors_list)):
                  if random.random() < probability:
                      self.deflectors_list[i][0] = random.randint(0, len(self.deflectors_list)-1)

            for i in range(len(self.deflectors_list)):
                 if random.random() < probability:
                     self.deflectors_list[random.randint(0, len(self.deflectors_list)-1)][1] = random.choice(self.output)

            for i in range(len(self.alphabet_directions)):
                 if random.random() < probability:
                     self.alphabet_directions[i] = 1 if self.alphabet_directions[i] == -1 else -1

            for i in range(len(self.alphabet_start_positions)):
                  if random.random() < probability:
                      self.alphabet_start_positions[i] = random.randint(0, len(self.deflectors_list)-1)

            if random.random() < probability:
                  self.set_holes(self.output, len(self.deflectors_list), self.holes_multiplier)

        return self

    def train_continuous(self, test_cases, epochs=300, lr=0.05, batch_size=16,
                          temp_start=2.5, temp_end=0.5, sharpness=0.75,
                          repeat_penalty_weight=0.3,
                          curriculum=False, curriculum_start_size=4, curriculum_growth=4,
                          curriculum_threshold=0.9, curriculum_check_every=20,
                          warm_start=True, commit=True, verbose=True):
        """
        Continuous relaxation training.

        Every discrete choice the brain makes is replaced with a differentiable
        stand-in for the duration of training:
          - each deflector's destination index      -> a softmax over all deflectors
          - each deflector's hole/bounce identity    -> a sigmoid (hole probability)
          - each deflector's output symbol (if hole) -> a softmax over self.output
          - each deflector's bounce value (if not)   -> a free real-valued scalar
          - each alphabet direction (+-1)            -> tanh of a free scalar
          - each alphabet start position              -> a softmax over all deflectors
          - ball_energy (the ball's starting energy)  -> a free real-valued scalar

        These cover every field Mutate touches on the discrete brain, plus
        ball_energy. Three other brain-level attributes are deliberately left
        untouched, because they don't have a meaningful continuous form or
        never reach the loss at all:
          - `holes_multiplier` only controls how many passes set_holes runs
            during __init__/Mutate (discrete, GA-side bookkeeping) -- it never
            appears in ricochet/compute_input or in _c_simulate below, so
            there's no forward computation for a gradient to flow through.
          - `alphabet` and `output` are the literal vocabularies (which
            characters exist), not values assigned to them -- everything
            about *how* each character/symbol is used (direction, start
            position, which symbol a hole emits) is already covered above.
            Their sizes (`len(alphabet)`, `len(output)`) fix the shapes of
            the tensors above; changing their contents would be a structural/
            architecture change (a different vocabulary entirely), not a
            parameter value to relax and optimize.

        The ricochet loop is unrolled for `cap` steps and the hard indexing /
        branching / while-loop of the discrete simulator is replaced by
        probability-weighted ("expected value") versions of the same
        arithmetic, so the whole forward pass is differentiable and can be
        trained with gradient descent against the task loss instead of the
        genetic algorithm's mutate-and-select loop. A temperature is annealed
        from `temp_start` down to `temp_end` over training so the softmax
        distributions sharpen towards one-hot (i.e. towards the original
        discrete model) as training progresses.

        Because the discrete model can emit a variable number of output
        symbols per input, this relaxation makes one simplifying assumption:
        it treats the *first hole reached* by each character's trajectory as
        that character's contribution to the output, and aligns those
        per-character predictions against the target string position by
        position, right-aligned (the last trajectory position lines up with
        the last target character, and so on).

        Every trajectory position is trained, not just the aligned ones.
        `never_fired = 1 - out_dist.sum()` (the probability a trajectory ends
        the simulation without ever emitting anything) turns each position's
        prediction into a genuine (K+1)-way distribution: the K real output
        symbols, plus one "emit nothing" class, all summing to 1 by
        construction. Leading input positions that fall outside the aligned
        window (i.e. that have no corresponding target character at all) are
        given "emit nothing" as their target class in that same distribution.
        Earlier versions of this method left those leading positions with no
        training signal whatsoever, so they were free to fire and leak
        arbitrary symbols into the real, full output regardless of how low
        the loss on the aligned positions got; a later attempt fixed that
        with a second, separately-weighted penalty term, but that fought the
        main loss for control of shared deflectors and could destabilize
        training. Folding "should this position emit at all" into the *same*
        per-position cross-entropy as the main task avoids both problems --
        there's one loss, and every position (aligned or not) has a
        well-defined target under it.

        There's a related, narrower pathology this doesn't cover: once a
        trajectory reaches a hole, this objective stops caring what happens
        for the rest of that trajectory's `cap` steps. In the real, discrete
        brain, if that trajectory keeps landing on holes afterward (a direct
        self-loop, a 2-cycle between two holes, or any longer cycle -- all
        equally invisible to the "first hole" objective above), `ricochet`
        just keeps re-appending symbols for the rest of the `cap` steps (e.g.
        `"3333333333"`, `"2323232323"`). `repeat_penalty_weight` (default 0.3)
        penalizes the expected number of *additional* hole-hits after the
        first one for every trajectory in the batch (`extra_fire` from
        `_c_simulate`) -- computed from the same per-example forward pass
        already used for the main loss, so it directly targets whatever
        cycle each specific trajectory is actually at risk of, rather than
        only guarding against direct self-loops.

        The trained continuous parameters are kept on this instance (usable
        via `predict_continuous` / `evaluate_continuous`, see below) whether
        or not you discretize. If `commit=True` (the default), every
        continuous parameter is *also* discretized (argmax / sign / rounding)
        and written directly back into this instance's own `deflectors_list`,
        `alphabet_directions` and `alphabet_start_positions` attributes, so
        every other method on this class (compute_input, ricochet, clone,
        Mutate, display_deflectors, ...) keeps working exactly as before,
        just with gradient-trained values in place of the old ones. Pass
        `commit=False` to leave those discrete attributes untouched and only
        keep the trained continuous version (e.g. if you want to compare
        continuous vs. discrete behavior, or keep iterating with
        `train_continuous` again before ever committing to a discrete brain).

        Returns `(loss_history, accuracy_history)`, both per-epoch lists of
        floats. `accuracy_history` is a cheap "token accuracy" over *every*
        trajectory position in each epoch's batch (aligned or blank): it
        checks whether that position's predicted class -- its top symbol if
        its emit-probability is >= 0.5, else "blank" -- matches the target
        class, and reports the fraction that do. It's computed for free from
        the same numbers already used for the loss -- no extra simulation is
        run -- so while it now also reflects whether leading positions are
        correctly suppressed, it's still not the same as full-sequence
        exact-match correctness on the real, discrete brain (see
        `evaluate_continuous` / `evaluate_performance` for that).

        `curriculum=True` enables curriculum learning: instead of sampling every
        batch from all of `test_cases`, training starts on a small "active"
        subset (`curriculum_start_size` examples) and grows it as the model
        actually masters what it currently has. Every `curriculum_check_every`
        epochs, the real exact-match accuracy on the active subset is measured
        with `evaluate_continuous` (not the cheap `token_acc` proxy -- this is
        the honest, full-answer metric); once it's >= `curriculum_threshold`,
        `curriculum_growth` more examples are added to the active subset (in
        the order `test_cases` was given, or shuffled once up front if you
        pass a shuffled list) and training continues, until the active subset
        covers all of `test_cases`. This runs the periodic real-accuracy check
        against a small, growing pool, so it stays cheap even though it's not
        the free `token_acc` proxy. If a `verbose` print is happening the same
        epoch as an expansion, the curriculum message prints alongside it.
        """
        import torch

        self._init_continuous_params(warm_start=warm_start, sharpness=sharpness)
        c = self._c_meta
        output_index = c['output_index']

        optimizer = torch.optim.Adam(
            [self._c_dest_logits, self._c_hole_logit, self._c_symbol_logits,
             self._c_bounce_val, self._c_direction_param, self._c_start_logits,
             self._c_ball_energy], lr=lr
        )

        if curriculum:
            active_pool = list(test_cases)
            active_n = min(curriculum_start_size, len(active_pool))
            if verbose:
                print(f"[curriculum] starting with {active_n}/{len(active_pool)} examples")
        else:
            active_pool = None
            active_n = None

        history = []
        acc_history = []
        temp = temp_start
        for epoch in range(epochs):
            frac = epoch / max(1, epochs - 1)
            temp = temp_start + (temp_end - temp_start) * frac

            if curriculum:
                pool = active_pool[:active_n]
                if epoch > 0 and epoch % curriculum_check_every == 0 and active_n < len(active_pool):
                    acc = self.evaluate_continuous(pool, temp=temp, ratio=False)
                    if verbose:
                        print(f"[curriculum] epoch {epoch}: active set {active_n}/{len(active_pool)} "
                              f"exact-match={acc:.3f}")
                    if acc >= curriculum_threshold:
                        active_n = min(active_n + curriculum_growth, len(active_pool))
                        pool = active_pool[:active_n]
                        if verbose:
                            print(f"[curriculum] epoch {epoch}: mastered current set -- "
                                  f"expanding to {active_n}/{len(active_pool)} examples")
            else:
                pool = test_cases

            batch = random.sample(pool, min(batch_size, len(pool)))

            optimizer.zero_grad()
            total_loss = torch.tensor(0.0)
            count = 0
            correct = 0
            repeat_loss = torch.tensor(0.0)
            repeat_count = 0
            for inp, target in batch:
                tokens = TokenizeInput(inp, self.alphabet)
                indices = [self.alphabet.index(t) for t in tokens if t in self.alphabet]
                if not indices:
                    continue
                outs, never_fired, extra_fire = self._c_simulate(indices, temp)
                target_indices = [output_index[ch] for ch in target if ch in output_index]
                L = len(outs)
                n = min(L, len(target_indices))
                # Right-align: the *last* n trajectory positions correspond to the *last* n
                # target characters. Test inputs typically share a literal, invariant prefix
                # (e.g. every "1+i" starts with the same "1+"), so left-aligning would
                # repeatedly score that shared, input-invariant trajectory against different
                # targets -- gradients then can't distinguish examples at all. The trailing
                # characters of the input are where the example-specific information (e.g.
                # the digits of i) actually lives, so align from the end instead.
                out_offset = L - n
                tgt_offset = len(target_indices) - n
                # Score EVERY position, not just the aligned tail. `out_dist[k]` and
                # `never_fired` already form one consistent (K+1)-way distribution over
                # "emit symbol k" / "emit nothing" (never_fired = 1 - out_dist.sum() by
                # construction), so positions before out_offset -- which have no target
                # character at all -- are simply given "emit nothing" as their target class
                # in that SAME distribution, rather than being left with no training signal
                # (which is what let them leak arbitrary symbols into the real output) or
                # fought over with a separately-weighted penalty term (which destabilized
                # training by competing against the main loss on shared deflectors).
                for i in range(L):
                    if i < out_offset:
                        target_prob = never_fired[i]
                        target_class = c['K']   # the "blank" class
                    else:
                        target_idx = target_indices[tgt_offset + (i - out_offset)]
                        target_prob = outs[i][target_idx]
                        target_class = target_idx
                    total_loss = total_loss - torch.log(target_prob + 1e-8)

                    p_emit = float(outs[i].sum().item())
                    pred_class = int(torch.argmax(outs[i]).item()) if p_emit >= 0.5 else c['K']
                    if pred_class == target_class:
                        correct += 1
                    count += 1

                    repeat_loss = repeat_loss + extra_fire[i]
                    repeat_count += 1

            if count == 0:
                continue
            task_loss = total_loss / count
            train_acc = correct / count
            combined_loss = task_loss
            if repeat_penalty_weight and repeat_count:
                combined_loss = combined_loss + repeat_penalty_weight * (repeat_loss / repeat_count)
            combined_loss.backward()
            optimizer.step()
            history.append(task_loss.item())
            acc_history.append(train_acc)

            if verbose and (epoch % max(1, epochs // 20) == 0 or epoch == epochs - 1):
                print(f"[continuous] epoch {epoch}: loss={task_loss.item():.4f} token_acc={train_acc:.3f} temp={temp:.2f}")

        # Remember the temperature training ended at, so predict_continuous /
        # evaluate_continuous default to something close to what was actually trained.
        self._c_meta['last_temp'] = temp

        if commit:
            self._discretize_continuous_into(self)

        return history, acc_history

    def _init_continuous_params(self, warm_start=True, sharpness=0.75):
        """Build (or rebuild) this instance's continuous relaxation parameters from its
        current discrete state, storing them on the instance so they persist beyond a
        single train_continuous call and can be used by predict_continuous /
        evaluate_continuous / _discretize_continuous_into."""
        import torch

        D = len(self.deflectors_list)
        A = len(self.alphabet)
        K = len(self.output)
        output_index = {sym: k for k, sym in enumerate(self.output)}
        positions = torch.arange(D, dtype=torch.float32)
        abs_diff = (positions.unsqueeze(0) - positions.unsqueeze(1)).abs()  # abs_diff[d, d'] = |d - d'|

        CONF = 4.0  # logit magnitude used so a warm-started softmax closely matches the current discrete choice
        dest_logits = torch.zeros(D, D)
        hole_logit = torch.zeros(D)
        symbol_logits = torch.zeros(D, K)
        bounce_val = torch.zeros(D)
        direction_param = torch.zeros(A)
        start_logits = torch.zeros(A, D)
        ball_energy = torch.tensor(float(self.ball_energy))

        if warm_start:
            for i, (dest, val) in enumerate(self.deflectors_list):
                dest_logits[i, dest] = CONF
                if isinstance(val, int):
                    hole_logit[i] = -CONF
                    bounce_val[i] = float(val)
                else:
                    hole_logit[i] = CONF
                    symbol_logits[i, output_index[val]] = CONF
            for i, d in enumerate(self.alphabet_directions):
                direction_param[i] = CONF if d == 1 else -CONF
            for i, p in enumerate(self.alphabet_start_positions):
                start_logits[i, p] = CONF
        else:
            dest_logits.normal_(0, 0.1)
            hole_logit.normal_(0, 0.1)
            symbol_logits.normal_(0, 0.1)
            bounce_val.normal_(0, 1.0)
            direction_param.normal_(0, 0.1)
            start_logits.normal_(0, 0.1)

        for p in (dest_logits, hole_logit, symbol_logits, bounce_val, direction_param, start_logits, ball_energy):
            p.requires_grad_(True)

        self._c_dest_logits = dest_logits
        self._c_hole_logit = hole_logit
        self._c_symbol_logits = symbol_logits
        self._c_bounce_val = bounce_val
        self._c_direction_param = direction_param
        self._c_start_logits = start_logits
        self._c_ball_energy = ball_energy
        self._c_meta = dict(D=D, A=A, K=K, cap=10,
                             output_index=output_index, positions=positions, abs_diff=abs_diff,
                             sharpness=sharpness, last_temp=None)

    def _c_soft_index_dist(self, travel_value, temp):
        import torch
        import torch.nn.functional as F
        c = self._c_meta
        diff = c['positions'] - travel_value
        diff = torch.remainder(diff + c['D'] / 2, c['D']) - c['D'] / 2
        logits = -(diff ** 2) / (c['sharpness'] * temp)
        return F.softmax(logits, dim=0)

    def _c_simulate(self, input_indices, temp):
        """Run the differentiable, soft ricochet simulation using this instance's
        persisted continuous parameters (self._c_*). Returns (outputs, never_fired,
        extra_fire), one entry per input character: `outputs[i]` is a probability
        distribution over self.output (via the "first hole reached" convention -- see
        train_continuous's docstring), `never_fired[i]` is the probability that
        trajectory never emits anything within `cap` steps, and `extra_fire[i]` is the
        expected number of *additional* hole-hits after the first one (i.e. how much
        the trajectory keeps re-firing instead of settling down for the rest of its
        cap steps)."""
        import torch
        import torch.nn.functional as F

        c = self._c_meta
        dest_logits, hole_logit = self._c_dest_logits, self._c_hole_logit
        symbol_logits, bounce_val = self._c_symbol_logits, self._c_bounce_val
        direction_param, start_logits = self._c_direction_param, self._c_start_logits

        L = len(input_indices)
        directions_soft = torch.tanh(direction_param)              # (A,) relaxation of +-1
        dirs = directions_soft[input_indices]                      # (L,)
        total_dir = dirs.sum()

        start_probs_all = F.softmax(start_logits / temp, dim=1)    # (A, D)
        expected_start_all = (start_probs_all * c['positions']).sum(dim=1)  # (A,)
        starts = expected_start_all[input_indices]                 # (L,)

        dest_probs = F.softmax(dest_logits / temp, dim=1)          # (D, D)
        hole_p = torch.sigmoid(hole_logit)                         # (D,)
        symbol_probs = F.softmax(symbol_logits / temp, dim=1)      # (D, K)
        weighted_abs = (dest_probs * c['abs_diff']).sum(dim=1)     # (D,) expected hop distance

        outputs = []
        never_fired = []
        extra_fire = []
        for i in range(L):
            travel_i = starts[i] + (total_dir - dirs[i])           # matches Brain.travel()
            state = self._c_soft_index_dist(travel_i, temp)        # distribution over current deflector
            be = self._c_ball_energy
            alive = torch.tensor(1.0)          # soft "be > 0 and cap_idx < cap" gate
            not_hole_yet = torch.tensor(1.0)   # soft "no hole reached yet" gate
            out_dist = torch.zeros(c['K'])
            extra = torch.tensor(0.0)          # expected re-fires after the first hole

            for _ in range(c['cap']):
                p_hole = (state * hole_p).sum()
                p_bounce_energy = (state * (1 - hole_p) * bounce_val).sum()
                exp_dist = (state * weighted_abs).sum()

                already_fired = alive * (1.0 - not_hole_yet)
                extra = extra + already_fired * p_hole

                p_first_hole = alive * not_hole_yet * p_hole
                sym_dist = (state.unsqueeze(1) * hole_p.unsqueeze(1) * symbol_probs).sum(dim=0)
                sym_dist = sym_dist / (p_hole + 1e-8)
                out_dist = out_dist + p_first_hole * sym_dist

                energy_delta = p_hole * 1.0 + p_bounce_energy - exp_dist
                be = be + energy_delta
                alive = alive * torch.sigmoid(be * 2.0)
                not_hole_yet = not_hole_yet * (1 - p_hole)

                state = state @ dest_probs

            outputs.append(out_dist)
            never_fired.append(1.0 - out_dist.sum())
            extra_fire.append(extra)

        return outputs, never_fired, extra_fire

    def _discretize_continuous_into(self, target_brain):
        """Argmax/sign/round this instance's continuous parameters (self._c_*) and write
        them into target_brain's discrete attributes (deflectors_list, alphabet_directions,
        alphabet_start_positions, ball_energy)."""
        import torch
        c = self._c_meta
        with torch.no_grad():
            for i in range(c['D']):
                dest = int(torch.argmax(self._c_dest_logits[i]).item())
                if torch.sigmoid(self._c_hole_logit[i]).item() > 0.5:
                    val = self.output[int(torch.argmax(self._c_symbol_logits[i]).item())]
                else:
                    val = int(round(self._c_bounce_val[i].item()))
                target_brain.deflectors_list[i][0] = dest
                target_brain.deflectors_list[i][1] = val
            for i in range(c['A']):
                target_brain.alphabet_directions[i] = 1 if self._c_direction_param[i].item() > 0 else -1
                target_brain.alphabet_start_positions[i] = int(torch.argmax(self._c_start_logits[i]).item())
            # ball_energy was originally always an even integer (bounce*2 from __init__), but
            # nothing downstream actually requires that -- ricochet's energy comparisons/updates
            # work fine with any real value -- so it's rounded to the nearest int here purely for
            # consistency with how the rest of the discrete brain's numbers look, not correctness.
            target_brain.ball_energy = int(round(self._c_ball_energy.item()))

    def commit_continuous(self):
        """Discretize this instance's already-trained continuous parameters (from a prior
        train_continuous(..., commit=False) call) into its discrete deflectors_list /
        alphabet_directions / alphabet_start_positions, without training any further."""
        if not hasattr(self, '_c_meta'):
            raise RuntimeError("No continuous parameters to commit -- call train_continuous(...) first.")
        self._discretize_continuous_into(self)

    def predict_continuous(self, input_str, temp=None, return_probs=False):
        """
        Answer `input_str` using the *continuous* (un-discretized) brain directly --
        no argmax/rounding of parameters, no dependency on deflectors_list/
        alphabet_directions/alphabet_start_positions at all. Requires a prior
        train_continuous(...) call (with or without commit=True) on this instance.

        Unlike training (which only scores a target-aligned tail of each trajectory),
        this decodes *every* input character's trajectory: if its "never fire" probability
        is below 0.5 it's treated as emitting, and the most likely symbol from its output
        distribution is appended, in the same left-to-right order compute_input uses.

        `temp` defaults to the temperature training ended at (colder = closer to the
        discretized/deterministic behavior; pass a higher temp to see softer, more
        exploratory behavior). Set `return_probs=True` to also get back, for every input
        character, `(probability it emits at all, {symbol: probability})`.
        """
        import torch

        if not hasattr(self, '_c_meta'):
            raise RuntimeError(
                "No continuous parameters on this brain yet -- call train_continuous(...) "
                "first (pass commit=False if you don't want it to also overwrite the "
                "discrete deflectors_list/alphabet_directions/alphabet_start_positions)."
            )
        c = self._c_meta
        if temp is None:
            temp = c['last_temp'] if c['last_temp'] is not None else 0.5

        tokens = TokenizeInput(input_str, self.alphabet)
        indices = [self.alphabet.index(t) for t in tokens if t in self.alphabet]
        if not indices:
            return ("", []) if return_probs else ""

        with torch.no_grad():
            outs, never_fired, _extra_fire = self._c_simulate(indices, temp)

        decoded = []
        probs = []
        for out_dist, nf in zip(outs, never_fired):
            p_emit = 1.0 - nf.item()
            sym_probs = {sym: p for sym, p in zip(self.output, out_dist.tolist())}
            probs.append((p_emit, sym_probs))
            if p_emit >= 0.5:
                decoded.append(self.output[int(torch.argmax(out_dist).item())])

        result = "".join(decoded)
        return (result, probs) if return_probs else result

    def evaluate_continuous(self, test_cases, temp=None, ratio=False):
        """Like evaluate_performance, but scores predict_continuous (the un-discretized
        brain) against test_cases instead of compute_input (the discretized/discrete
        brain). Returns the average score (0/1 exact-match per case, or a 0-1 similarity
        ratio if ratio=True)."""
        total = 0.0
        for inp, answer in test_cases:
            pred = self.predict_continuous(inp, temp=temp)
            if ratio:
                total += difflib.SequenceMatcher(None, pred, answer).ratio()
            else:
                total += 1 if pred == answer else 0
        return total / len(test_cases)

## Test case generation

In [ ]:
import random
import string
def generate_test_cases(amount, number, indiv=True):

  test_cases = []
  if indiv == True:
    test_cases = [(f'1+{i}', f"{random.randint(0, 1000)}") for i in range(amount)]
  else:
    test_cases = [(f'1+{i}', f'{1+i}') for i in range(amount)]

  return test_cases

## Fitness evaluation

In [ ]:
"""
Fitness evaluation module for Ball_Trap_Brain genetic algorithm.

This module contains functions to evaluate the performance of individual brains
and entire populations against test cases using string similarity metrics.
"""

import difflib
import math


def evaluate_performance(input_val, answer, brain, ratio=False):

    output = brain.compute_input(input_val)
    if ratio:
        return difflib.SequenceMatcher(None, ''.join(output), answer).ratio()
    else:
      return 1 if ''.join(output) == answer else 0


import concurrent.futures

def eval_single_brain(args):
    individual, test_cases, ratio = args
    avg_fitness = sum(
        evaluate_performance(inp, ans, individual, ratio) for inp, ans in test_cases
    ) / len(test_cases)
    return avg_fitness

def evaluate_population_parallel(population, test_cases, ratio=False):
    # Package arguments for the worker pool
    args = [(ind, test_cases, ratio) for ind in population]

    # Run across available CPU cores
    with concurrent.futures.ProcessPoolExecutor() as executor:
        results = list(executor.map(eval_single_brain, args))

    best_fitness = max(results)
    best_brain = population[results.index(best_fitness)]
    return best_fitness, best_brain

## Genetic algorithm (`main`)

Still here and unchanged, for training single (non-council) brains the original way if you want to.

In [ ]:
"""
Genetic algorithm to evolve Ball_Trap_Brain instances to produce desired outputs.

This module runs a genetic algorithm that evolves a population of brains over
multiple generations. Each brain is evaluated against test cases, and the best
performer is mutated to create the next generation.
"""

string = "This module runs a genetic algorithm that evolves a population of brains over multiple generations. Each brain is evaluated against test cases, and the best performer is mutated to create the next generation."

import copy
import difflib
import random
import shelve

# Test cases: pairs of (input_string, expected_output_string) used for fitness evaluation


alphabet = [f'{i}' for i in range(10)] + ['+'] #+ list("abcdefghijklmnopqrstuvwxyz")  # Define the alphabet used in the brains
output = [f'{i}' for i in range(10)]

def main(mini_batch, alphabet, generations=500, pop_size=100, brain=Ball_Trap_Brain(1000, 4, alphabet, output, 2), ratio=False):
    """
    Run the genetic algorithm for evolving Ball_Trap_Brain instances.

    Returns:
        Ball_Trap_Brain: The best brain found after evolution.
    """
    # Evolution hyperparameters
    population_size = pop_size

    # Initialize population with random brains
    # Parameters: 10 deflectors, 2 bounce energy, alphabet of 6 characters

    population = [brain.clone().Mutate(1.0, 0.025) for _ in range(population_size)]
    population.append(brain)
    best_individual = None
    mini_batch_0 = copy.copy(mini_batch)
    #a = [mini_batch_0.pop(0) for i in range(10)]
    highest_score = -1
    # Run evolution loop
    for generation in range(generations):

        # Evaluate all brains and find the best performer
        best_fitness, best_individual = evaluate_population_parallel(population,mini_batch, ratio)



        # Check for perfect solution
        if best_fitness >= 1:
          print(f"Generation {generation}: Solution found!")
          return best_individual
          #for i in range(10):
            #a.append(mini_batch_0.pop(0))


        # Create next generation by mutating the best brain multiple times
        # New code: passes best_fitness, individual_probability, base_probability
        new_population = [best_individual.clone().Mutate(1.0, 0.025) for _ in range(population_size)]
        population = new_population
        population.append(best_individual)
        print(f"Generation {generation}: Best fitness = {best_fitness}")
    return best_individual

training_1 = generate_test_cases(amount=100, number='1', indiv=True)

## `Council_Brain` & council training — continuous

`train_council` trains each member with `train_continuous(..., commit=False)`, so every council member is a brain that only has trained *continuous* parameters -- its discrete `deflectors_list` is never touched.

`Council_Brain.compute_input` routes each member to `predict_continuous` automatically whenever it detects trained continuous parameters (`hasattr(brain, '_c_meta')`), and falls back to the discrete `compute_input` otherwise -- so a council can still mix genetic-algorithm-trained or `commit_continuous()`-ed members with continuous ones if you want.

In [ ]:
from collections import Counter

class Council_Brain:
    def __init__(self, brains):
        """
        Initializes the council with a list of separately trained brains.
        """
        self.brains = brains

    def compute_input(self, input_val):
        """
        Each brain processes the input separately, then the council votes on the
        final answer by majority.

        A brain that has been trained with train_continuous(..., commit=False)
        (the default for train_council below) only has trained *continuous*
        parameters -- its discrete deflectors_list/etc are untouched, essentially
        random, and would give meaningless answers -- so such brains are asked via
        predict_continuous instead of compute_input. Brains without continuous
        parameters (e.g. genetic-algorithm-trained, or committed with
        commit_continuous()) still use compute_input as before, so a council can
        freely mix either kind of member.
        """
        votes = []
        for brain in self.brains:
            if hasattr(brain, '_c_meta'):
                output_str = brain.predict_continuous(input_val)
            else:
                # Each brain outputs a list of characters; join them to form the string
                output_str = ''.join(brain.compute_input(input_val))
            votes.append(output_str)

        # Handle empty cases securely
        if not votes:
            return []

        # Tally the votes and find the majority
        vote_counts = Counter(votes)
        majority_vote = vote_counts.most_common(1)[0][0]

        # Return as a list of characters to perfectly match the original Ball_Trap_Brain format
        return list(majority_vote)

def train_council(num_brains, training_data, alphabet, output_format,
                   epochs=400, lr=0.15, batch_size=16, deflectors=100, bounce=10):
    """
    Trains a council where each brain is trained separately using continuous
    relaxation gradient descent (train_continuous) instead of the genetic
    algorithm. Each member keeps its trained continuous parameters
    (commit=False), and Council_Brain.compute_input above automatically routes
    to predict_continuous for members trained this way -- no discretization
    step is needed anywhere in this pipeline.
    """
    trained_brains = []

    for i in range(num_brains):
        print(f"\n=====================================")
        print(f" Training Council Brain {i+1} / {num_brains}")
        print(f"=====================================")
        base_brain = Ball_Trap_Brain(deflectors, bounce, alphabet, output_format, 2)

        base_brain.train_continuous(
            training_data,
            epochs=epochs,
            lr=lr,
            batch_size=batch_size,
            commit=False,
            verbose=True,
        )
        trained_brains.append(base_brain)

    return Council_Brain(trained_brains)

## Continuous relaxation training — single-brain demo

Trains one brain with gradient descent (requires `torch`; installed via `pip install torch` if missing) with `commit=False`, so the discrete brain is left completely untouched, and answers questions directly with the continuous version via `predict_continuous`. Then calls `commit_continuous()` and compares against the real, discrete `compute_input`.

In [ ]:
# !pip install torch --quiet   # uncomment if torch isn't already installed

import random
random.seed(0)

demo_brain = Ball_Trap_Brain(70, 4, alphabet, output, 1)
demo_cases = generate_test_cases(amount=30, number='1', indiv=False)  # '1+i' -> str(1+i)

loss_history, acc_history = demo_brain.train_continuous(
    demo_cases,
    epochs=200,
    lr=0.15,
    batch_size=16,
    warm_start=True,
    commit=False,   # keep the discrete brain untouched; only train the continuous version
    verbose=True,
)
print('token_acc[0] ->', acc_history[0], ' token_acc[-1] ->', acc_history[-1])

print()
print('=== CONTINUOUS predictions ===')
for inp, ans in demo_cases[:10]:
    pred = demo_brain.predict_continuous(inp)
    print(inp, '->', repr(pred), ' (target', ans, ') match:', pred == ans)
cont_exact = demo_brain.evaluate_continuous(demo_cases, ratio=False)
cont_ratio = demo_brain.evaluate_continuous(demo_cases, ratio=True)
print('continuous exact-match:', cont_exact, ' ratio:', cont_ratio)

demo_brain.commit_continuous()
print()
print('=== DISCRETE predictions (after commit_continuous) ===')
for inp, ans in demo_cases[:10]:
    pred = ''.join(demo_brain.compute_input(inp))
    print(inp, '->', repr(pred), ' (target', ans, ') match:', pred == ans)
disc_exact = sum(evaluate_performance(i, a, demo_brain, ratio=False) for i, a in demo_cases) / len(demo_cases)
disc_ratio = sum(evaluate_performance(i, a, demo_brain, ratio=True) for i, a in demo_cases) / len(demo_cases)
print('discrete exact-match:', disc_exact, ' ratio:', disc_ratio)

[continuous] epoch 0: loss=1.8848 token_acc=0.561 temp=2.50
[continuous] epoch 10: loss=1.5519 token_acc=0.525 temp=2.40
[continuous] epoch 20: loss=1.2660 token_acc=0.621 temp=2.30
[continuous] epoch 30: loss=1.0185 token_acc=0.650 temp=2.20
[continuous] epoch 40: loss=0.8501 token_acc=0.644 temp=2.10
[continuous] epoch 50: loss=0.7604 token_acc=0.655 temp=2.00
[continuous] epoch 60: loss=0.6951 token_acc=0.707 temp=1.90
[continuous] epoch 70: loss=0.6610 token_acc=0.746 temp=1.80
[continuous] epoch 80: loss=0.6210 token_acc=0.759 temp=1.70
[continuous] epoch 90: loss=0.6092 token_acc=0.770 temp=1.60
[continuous] epoch 100: loss=0.5185 token_acc=0.800 temp=1.49
[continuous] epoch 110: loss=0.5467 token_acc=0.783 temp=1.39
[continuous] epoch 120: loss=0.5412 token_acc=0.759 temp=1.29
[continuous] epoch 130: loss=0.5671 token_acc=0.746 temp=1.19
[continuous] epoch 140: loss=0.4871 token_acc=0.817 temp=1.09


KeyboardInterrupt: 

## Curriculum learning — demo

Same single-brain training, but starting on just 4 examples and growing the active set by 4 whenever real exact-match accuracy on the current active set reaches 70%. Watch the `[curriculum]` lines: the active set size grows over training instead of using all 30 examples from epoch 0.

In [ ]:
# import random
# random.seed(0)

# curriculum_brain = Ball_Trap_Brain(30, 4, alphabet, output, 2)
# curriculum_cases = generate_test_cases(amount=30, number='1', indiv=False)

# loss_history, acc_history = curriculum_brain.train_continuous(
#     curriculum_cases,
#     epochs=250,
#     lr=0.15,
#     batch_size=8,
#     curriculum=True,
#     curriculum_start_size=4,
#     curriculum_growth=4,
#     curriculum_threshold=0.7,
#     curriculum_check_every=25,
#     commit=False,
#     verbose=True,
# )

# print()
# print('Final continuous exact-match on the FULL 30-example set:',
#       curriculum_brain.evaluate_continuous(curriculum_cases, ratio=False))

## Continuous council training — demo

Trains a small council (a handful of brains, few epochs, so it runs quickly here) with `train_council`, then predicts with the council's majority vote -- every member answers via `predict_continuous` under the hood.

In [ ]:
# import random
# random.seed(0)

# council_cases = generate_test_cases(amount=30, number='1', indiv=False)
# random.shuffle(council_cases)
# council_train = council_cases[:20]
# council_test = council_cases[20:]

# council = train_council(
#     num_brains=2,
#     training_data=council_train,
#     alphabet=alphabet,
#     output_format=output,
#     epochs=100,
#     lr=0.15,
#     batch_size=16,
#     deflectors=30,
#     bounce=4,
# )

# print()
# print('=== Council predictions (continuous vote) ===')
# for inp, ans in council_test:
#     pred = ''.join(council.compute_input(inp))
#     print(inp, '->', repr(pred), ' (target', ans, ') match:', pred == ans)

## Full council training run (optional, slow)

This reproduces the original script's `if __name__ == '__main__':` block: it trains a 50-brain council (via continuous relaxation) and evaluates it. This can take a long time, so it's guarded behind `RUN_FULL_TRAINING`.

In [ ]:
RUN_FULL_TRAINING = False  # set to True to actually run the full council training

if RUN_FULL_TRAINING:
    # Generate the test cases using the existing generation function
    test_cases = generate_test_cases(amount=1000, number='1', indiv=False)
    random.shuffle(test_cases)

    # Split the dataset
    training_0 = test_cases[:int(0.5 * len(test_cases))]
    training_test = test_cases[int(0.5 * len(test_cases)):]


    # Define council parameters
    COUNCIL_SIZE = 50      # Number of voting brains in the council
    EPOCHS = 400            # Continuous training epochs per brain
    BATCH_SIZE = 16         # Batch size per epoch

    # Train the council (each brain is trained entirely separately, via continuous relaxation)
    print(f"Initializing Council Training with {COUNCIL_SIZE} brains...")
    council = train_council(
        num_brains=COUNCIL_SIZE,
        training_data=training_0,
        alphabet=alphabet,
        output_format=output,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE
    )



    # Evaluate the entire council's performance against the testing set
    # Note: evaluate_population_parallel relies on a list of brains, so we wrap the council in a list
    print("\nEvaluating Council performance on test batch...")
    council_fitness, _ = evaluate_population_parallel([council], training_test, False)
    council_fitness_training, _ = evaluate_population_parallel([council], training_0, False)
    print(f"Overall Council Fitness: {council_fitness}, Council training fitness: {council_fitness_training}")